# R07 · Multicore Linux - affinity, interrupt routing, false sharing, Amdahl

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/05_realtime_linux/R07_multicore_linux_affinity_interrupt_routing_false.ipynb)

**Track:** Real-time and Linux · **Skill axis (sheet):** Real-time and Linux · **Time:** about 55 min

**Skill Hub concepts:** Symmetric multiprocessing; CPU affinity and core isolation; Interrupt affinity; Cache coherence and false sharing; Amdahl's law

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("R07")

## Map: one kernel, several identical cores

**Symmetric multiprocessing (SMP):** one OS schedules threads on any core; caches are kept coherent by hardware
(MESI-style protocols move 64-byte **cache lines** between cores).

| Tool | Interface | Use |
|---|---|---|
| Thread/process affinity | `sched_setaffinity`, `taskset -c 3 ./ctrl`, `pthread_setaffinity_np` | keep the control loop on one core (warm caches, no migration) |
| Core isolation | boot args `isolcpus=3 nohz_full=3 rcu_nocbs=3` | the scheduler puts nothing else on core 3 |
| Interrupt affinity | `/proc/irq/<n>/smp_affinity` (hex mask), `irqbalance` | route the NIC/camera interrupts away from the control core, or onto it if the loop consumes them |
| Observe | `/proc/interrupts` (per-CPU counts), `top -H` / `ps -eLo pid,psr` | see where work actually runs |

CPU mask: bit $k$ set means CPU $k$ allowed. CPUs {0, 2} -> `0b101` = `0x5`.

**False sharing:** two threads write different variables that sit in the **same cache line**; every write steals the
line from the other core, so performance collapses although no data is shared. Fix: pad or align each hot variable
to its own line (`_Alignas(64)`, `alignas(std::hardware_destructive_interference_size)`).

**Amdahl's law:** with parallel fraction $p$ on $n$ cores, $S(n) = \dfrac{1}{(1 - p) + p/n}$, and
$S(\infty) = 1/(1 - p)$.

## Task 1: remove false sharing

In [ ]:
%%writefile counters.h
#include <stdatomic.h>
struct counters_bad { atomic_long a; atomic_long b; };          /* same 64-byte line */
struct counters_good { atomic_long a; atomic_long b; };      /* TODO: give each counter its own cache line */

In [ ]:
lab.check("padding")

In [ ]:
%%writefile bench.c
#define _GNU_SOURCE
#include <pthread.h>
#include <stdio.h>
#include <time.h>
#include "counters.h"
#define N 20000000L
static struct counters_bad bad; static struct counters_good good;
static void *inc(void *p) { atomic_long *c = p; for (long i = 0; i < N; i++) atomic_fetch_add_explicit(c, 1, memory_order_relaxed); return 0; }
static double run(atomic_long *x, atomic_long *y) {
    struct timespec a, b; pthread_t t1, t2;
    clock_gettime(CLOCK_MONOTONIC, &a);
    pthread_create(&t1, 0, inc, x); pthread_create(&t2, 0, inc, y);
    pthread_join(t1, 0); pthread_join(t2, 0);
    clock_gettime(CLOCK_MONOTONIC, &b);
    return (b.tv_sec - a.tv_sec) + (b.tv_nsec - a.tv_nsec) / 1e9;
}
int main(void) {
    double tb = run(&bad.a, &bad.b), tg = run(&good.a, &good.b);
    printf("same line: %.2f s   separate lines: %.2f s   speedup %.1fx\n", tb, tg, tb / tg);
}

In [ ]:
from roboprep import clab
import os
print("CPUs visible:", os.cpu_count(), "| this process may run on:", sorted(os.sched_getaffinity(0)))
print(clab.sh("gcc -O2 -pthread bench.c -o bench && ./bench", timeout=120).stdout)
print("(On a 2-vCPU Colab VM the two vCPUs may be hyperthreads of one core that share L1, which hides most of the effect.)")

## Task 2: masks, affinity and interrupt counts

In [ ]:
def cpu_mask(cpus):
    raise NotImplementedError  # TODO

def parse_mask(text):
    raise NotImplementedError  # TODO
    

def irq_counts(proc_interrupts, irq):
    """Per-CPU counts for one IRQ number from /proc/interrupts text."""
    raise NotImplementedError  # TODO

lab.check("affinity", cpu_mask, parse_mask, irq_counts)

In [ ]:
%%writefile r07_affinity.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/r07.h
// Grade it with lab.check("affinity", "r07_affinity.c"). For C++, name the file .cpp in both places.
#include "r07.h"

uint64_t cpu_mask(const int *cpus, size_t n) { (void)cpus; (void)n; return 0; /* TODO */ }

uint64_t parse_mask(const char *text) { (void)text; return 0; /* TODO: hex digits, skip commas */ }

int irq_counts(const char *text, int irq, uint64_t *counts, size_t cap) {
    (void)text; (void)irq; (void)counts; (void)cap;
    return -1; /* TODO: count CPU columns in the header, find the "<irq>:" line, read the counts */
}

In [ ]:
# lab.check("affinity", "r07_affinity.c")  # uncomment after writing the C/C++ cell above

In [ ]:
print(open("/proc/interrupts").read()[:800])

## Task 3: Amdahl

In [ ]:
def amdahl(p, n):
    raise NotImplementedError  # TODO

lab.check("amdahl", amdahl)

In [ ]:
%%writefile r07_amdahl.c
// C/C++ version of Task 3 (optional, same tests). Declarations: roboprep/c/include/r07.h
// Grade it with lab.check("amdahl", "r07_amdahl.c"). For C++, name the file .cpp in both places.
#include "r07.h"

double amdahl(double p, double n) { (void)p; (void)n; return 0; /* TODO */ }

In [ ]:
# lab.check("amdahl", "r07_amdahl.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Apply).** 90% of a perception pipeline parallelizes. Speedup on 4 cores (3 decimals)?

**M2 (Apply).** Integer value of the affinity mask for CPUs {0, 2}?

**M3 (Apply).** Upper bound on that pipeline's speedup with unlimited cores?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** Two threads update `stats.rx_count` and `stats.tx_count` (adjacent `long`s). Each thread only touches its
own counter, yet throughput drops 5x when they run on different cores. Cause?
- a) a data race
- b) false sharing: both counters live in one cache line that bounces between cores on every write
- c) the scheduler

**K2 (Evaluate).** On a 4-core robot computer, plan where the 1 kHz control thread, the camera pipeline, the network
interrupts and the logging go, and how you enforce it.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `r07.h`).
3. In `roboprep/checks/r07.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()